In [7]:
import pandas as pd
import pickle

# Load data
ratings = pd.read_csv('Ratings.csv')
books   = pd.read_csv('Books.csv', low_memory=False)

# Fix column names
ratings.columns = ['user_id', 'isbn', 'rating']
books.columns   = ['isbn', 'title', 'author', 'year', 
                   'publisher', 'img_s', 'img_m', 'img_l']

# Drop zeros
ratings = ratings[ratings['rating'] > 0]

print("Ratings:", ratings.shape)
print("Books:  ", books.shape)
print("✓ ready")

Ratings: (433671, 3)
Books:   (271360, 8)
✓ ready


In [8]:
def get_books_by_author(author_name, n=10):
    mask = books['author'].str.contains(
        author_name, case=False, na=False
    )
    result = books[mask].merge(
        ratings.groupby('isbn').agg(
            avg_rating  = ('rating', 'mean'),
            num_ratings = ('rating', 'count')
        ).reset_index(),
        on='isbn', how='left'
    )
    result = result[result['num_ratings'] >= 5]
    result = result.sort_values('avg_rating', ascending=False)
    return result[['title', 'author', 'avg_rating', 'img_m']].head(n)

# Test
print(get_books_by_author('Rowling'))

                                                title             author  \
55  Harry Potter and the Chamber of Secrets Postca...      J. K. Rowling   
30                    Harry Potter Und Der Feuerkelch  Joanne K. Rowling   
15            Harry Potter y el prisionero de Azkaban      J. K. Rowling   
45             Harry Potter et la chambre des secrets      J. K. Rowling   
29         Harry Potter und der Gefangene von Azkaban      J. K. Rowling   
25  Harry Potter and the Sorcerer's Stone (Book 1 ...      J. K. Rowling   
6        Harry Potter and the Goblet of Fire (Book 4)      J. K. Rowling   
11     Harry Potter and the Sorcerer's Stone (Book 1)      J. K. Rowling   
36         Harry Potter und die Kammer des Schreckens  Joanne K. Rowling   
9   Harry Potter and the Prisoner of Azkaban (Book 3)      J. K. Rowling   

    avg_rating                                              img_m  
55    9.869565  http://images.amazon.com/images/P/0439425220.0...  
30    9.600000  http://imag

In [9]:
def get_books_by_genre(genre, n=10):
    GENRE_KEYWORDS = {
        'Fantasy':     ['fantasy', 'magic', 'wizard', 'dragon', 'witch'],
        'Sci-Fi':      ['science fiction', 'space', 'galaxy', 'robot', 'alien'],
        'Mystery':     ['mystery', 'detective', 'murder', 'crime', 'killer'],
        'Romance':     ['love', 'romance', 'heart', 'passion', 'wedding'],
        'Classic':     ['classic', 'pride', 'prejudice', 'dickens', 'austen'],
        'Thriller':    ['thriller', 'danger', 'secret', 'spy', 'terror'],
        'Non-fiction': ['history', 'biography', 'memoir', 'true', 'guide'],
        'Adventure':   ['adventure', 'journey', 'quest', 'survival'],
    }
    
    keywords = GENRE_KEYWORDS.get(genre, [])
    if not keywords:
        return pd.DataFrame()
    
    pattern = '|'.join(keywords)
    mask = books['title'].str.contains(pattern, case=False, na=False)
    
    result = books[mask].merge(
        ratings.groupby('isbn').agg(
            avg_rating  = ('rating', 'mean'),
            num_ratings = ('rating', 'count')
        ).reset_index(),
        on='isbn', how='left'
    )
    result = result[result['num_ratings'] >= 5]
    result = result.sort_values('avg_rating', ascending=False)
    return result[['title', 'author', 'avg_rating', 'img_m']].head(n)

# Test both genres
print("=== Fantasy ===")
print(get_books_by_genre('Fantasy'))

print("\n=== Mystery ===")
print(get_books_by_genre('Mystery'))

=== Fantasy ===
                                                  title                author  \
731                          Moreta: Dragonlady of Pern        Anne McCaffrey   
1339                          Support Your Local Wizard           Diane Duane   
1633  Dungeon Master's Guide: Core Rulebook II (Dung...            Monte Cook   
1634  Monster Manual: Core Rulebook III (Dungeons &a...         Skip Williams   
1154                          Dragonquest Achille Cover        Anne Mccaffrey   
430   The Lion, the Witch and the Wardrobe (Full-Col...           C. S. Lewis   
788   Wizard of Oz Postcards in Full Color (Card Books)            Ted Menten   
291   It's A Magical World: A Calvin and Hobbes Coll...        Bill Watterson   
31                              Dragons of Summer Flame         Margaret Weis   
1170  Player's Handbook: Core Rulebook I (Dungeons &...  Not Applicable (Na )   

      avg_rating                                              img_m  
731     9.600000  http

In [10]:
import pickle

# Save both functions + books dataframe into one pickle
# So Flask API can load everything in one shot

knowledge_data = {
    'books': books,
    'ratings_agg': ratings.groupby('isbn').agg(
        avg_rating  = ('rating', 'mean'),
        num_ratings = ('rating', 'count')
    ).reset_index()
}

pickle.dump(knowledge_data, open('knowledge_model.pkl', 'wb'))

# Also save top books from popularity model
popularity = pd.read_csv('top_books.csv')
print(f"✓ knowledge_model.pkl saved")
print(f"✓ top_books.csv already exists")
print("\nYour home page data is ready:")
print(f"  Trending books:     {len(popularity)} books")
print(f"  Books for author search: {len(books)} books")
print(f"  Books for genre search:  {len(books)} books")


✓ knowledge_model.pkl saved
✓ top_books.csv already exists

Your home page data is ready:
  Trending books:     506 books
  Books for author search: 271360 books
  Books for genre search:  271360 books
